# Data Wrangling

In this notebook, I continued on the analysis of the data collected on the game: Arc Raiders through YouTube comments.

In [2]:
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta
from scipy import stats
from sklearn.preprocessing import StandardScaler

# For text preprocessing (for RoBERTa later)
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the dataset
data = pd.read_csv('data/comments_data.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 64975 entries, 0 to 64974
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   comment_id         64975 non-null  str    
 1   text               64920 non-null  str    
 2   comment_date       64975 non-null  str    
 3   author_hash        64975 non-null  str    
 4   last_updated_at    64975 non-null  str    
 5   video_id           64975 non-null  str    
 6   video_title        64975 non-null  str    
 7   video_date         64975 non-null  str    
 8   channel_id         64975 non-null  str    
 9   keyword_matched    64975 non-null  str    
 10  video_description  59576 non-null  str    
 11  char_count         64975 non-null  int64  
 12  word_count         64975 non-null  int64  
 13  avg_word_length    64975 non-null  float64
 14  has_url            64975 non-null  bool   
 15  has_mention        64975 non-null  bool   
 16  has_hashtag        64975 non-null

## 1. Data Cleaning

### 1.1. Handling Missing Values

In [4]:
data.isna().sum()

comment_id              0
text                   55
comment_date            0
author_hash             0
last_updated_at         0
video_id                0
video_title             0
video_date              0
channel_id              0
keyword_matched         0
video_description    5399
char_count              0
word_count              0
avg_word_length         0
has_url                 0
has_mention             0
has_hashtag             0
exclamation_count       0
question_count          0
emoji_count             0
newline_count           0
uppercase_ratio         0
language                0
day_of_week             0
day_num                 0
dtype: int64

In [5]:
data[data['video_description'].notna()]

,comment_id,text,comment_date,author_hash,last_updated_at,video_id,video_title,video_date,channel_id,keyword_matched,...,has_mention,has_hashtag,exclamation_count,question_count,emoji_count,newline_count,uppercase_ratio,language,day_of_week,day_num
0,UgzpbB7Lu4LS5Rxo7qB4AaABAg,What do you guys think of the current state of...,2026-02-25 00:20:10,736a02de6a38580b74a1f8560d1003761b6a95cba452a6...,2026-02-25 00:20:10.000000,WZFW9MYB-Sg,ARC Raiders had ONE JOB and still failed #arcr...,2026-02-25 00:17:43,UCSvdjvUxDI8sk4U9kJeueBw,Arc Raiders,...,False,False,0,3,0,0,0.047337,en,Wednesday,2
1,Ugw2ObGAQWwvxYHWlnp4AaABAg,Do a pull up. 😅😂😂😂😂😂,2026-02-25 07:08:48,eff17868636ab900521ef0830e48fd0127176caaaa20ff...,2026-02-25 07:08:48.000000,WZFW9MYB-Sg,ARC Raiders had ONE JOB and still failed #arcr...,2026-02-25 00:17:43,UCSvdjvUxDI8sk4U9kJeueBw,Arc Raiders,...,False,False,0,0,6,0,0.050000,it,Wednesday,2
2,UgyBrH4m8fsQD7ckHf14AaABAg,I don’t think you understand how bad Bungie ha...,2026-02-25 01:01:37,bd9075867b8640c062527e497bc4a5333aff263ed85fff...,2026-02-25 01:01:37.000000,WZFW9MYB-Sg,ARC Raiders had ONE JOB and still failed #arcr...,2026-02-25 00:17:43,UCSvdjvUxDI8sk4U9kJeueBw,Arc Raiders,...,False,False,0,0,0,0,0.028169,en,Wednesday,2
3,UgyBrH4m8fsQD7ckHf14AaABAg.ATcrCwqHB9rATd2Of6SgXp,I witnessed a bit of it on Destiny for some ti...,2026-02-25 02:48:05,736a02de6a38580b74a1f8560d1003761b6a95cba452a6...,2026-02-25 02:48:05.000000,WZFW9MYB-Sg,ARC Raiders had ONE JOB and still failed #arcr...,2026-02-25 00:17:43,UCSvdjvUxDI8sk4U9kJeueBw,Arc Raiders,...,False,False,0,0,0,0,0.044944,en,Wednesday,2
4,UgxaDk8Jj6wSINGygN94AaABAg,I almost just got ragebaited by a discord mod,2026-02-25 00:58:46,e9bf1117dd9c5924e85bc44c19be00890713543fefb954...,2026-02-25 00:58:46.000000,WZFW9MYB-Sg,ARC Raiders had ONE JOB and still failed #arcr...,2026-02-25 00:17:43,UCSvdjvUxDI8sk4U9kJeueBw,Arc Raiders,...,False,False,0,0,0,0,0.022222,en,Wednesday,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64970,UgxHwLtXpBAC5ujltFB4AaABAg.ATKMI4W7sIzATKZetFhSxY,😂😂.. Usually more uglier rats @StM ..maan u ga...,2026-02-17 13:13:59,befca9aab5c11ce39e89c4770218615bf6e27cc3e16b68...,2026-02-17 13:13:59.000000,1aUeLebr7WQ,Easy SOLO method to earn 44k+ for Destroy Tick...,2026-02-17 09:54:09,UCX0RKVCST6aTpM-mA7V_Rgg,#ARCRAIDERS,...,True,False,0,0,3,0,0.121739,en,Tuesday,1
64971,Ugwex2xCwxly1HXQZYF4AaABAg,Great vid brother gonna try this,2026-02-17 11:08:34,e000f21c1e88eb1a795702640fff87386566ea6d876f04...,2026-02-17 11:08:34.000000,1aUeLebr7WQ,Easy SOLO method to earn 44k+ for Destroy Tick...,2026-02-17 09:54:09,UCX0RKVCST6aTpM-mA7V_Rgg,#ARCRAIDERS,...,False,False,0,0,0,0,0.031250,en,Tuesday,1
64972,UgxBkJl4c5fW295BefN4AaABAg,so you cant crawl while downed to the exit and...,2026-02-17 10:13:08,1315a01027cf2a47c22af30113da6d4d00a900fde3614d...,2026-02-17 10:13:08.000000,1aUeLebr7WQ,Easy SOLO method to earn 44k+ for Destroy Tick...,2026-02-17 09:54:09,UCX0RKVCST6aTpM-mA7V_Rgg,#ARCRAIDERS,...,False,False,1,0,1,0,0.005917,en,Tuesday,1
64973,UgxBkJl4c5fW295BefN4AaABAg.ATKEyO4THzVATKHLgZ4XiO,"If you have time left in the match, you can cr...",2026-02-17 10:33:56,a2311b9084966a41d798a54349ea9075e62b965c9a8bd1...,2026-02-17 10:33:56.000000,1aUeLebr7WQ,Easy SOLO method to earn 44k+ for Destroy Tick...,2026-02-17 09:54:09,UCX0RKVCST6aTpM-mA7V_Rgg,#ARCRAIDERS,...,False,False,0,0,0,2,0.021645,en,Tuesday,1


In [6]:
# Filter out rows with missing video descriptions and comments
filtered_data = data[data['video_description'].notna() & data['text'].notna()]
filtered_data.info()

<class 'pandas.DataFrame'>
Index: 59526 entries, 0 to 64974
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   comment_id         59526 non-null  str    
 1   text               59526 non-null  str    
 2   comment_date       59526 non-null  str    
 3   author_hash        59526 non-null  str    
 4   last_updated_at    59526 non-null  str    
 5   video_id           59526 non-null  str    
 6   video_title        59526 non-null  str    
 7   video_date         59526 non-null  str    
 8   channel_id         59526 non-null  str    
 9   keyword_matched    59526 non-null  str    
 10  video_description  59526 non-null  str    
 11  char_count         59526 non-null  int64  
 12  word_count         59526 non-null  int64  
 13  avg_word_length    59526 non-null  float64
 14  has_url            59526 non-null  bool   
 15  has_mention        59526 non-null  bool   
 16  has_hashtag        59526 non-null  boo

In [7]:
# Even if not null, some video descriptions, text or video_title might be empty strings. Let's check for that.
#filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['text'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0).sum()
filtered_data = filtered_data[~(filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['text'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0))]
filtered_data.shape

(59526, 25)

### 1.2. Standardizing Datetime

In [8]:
data['comment_date'] = pd.to_datetime(data['comment_date'])
data['video_date'] = pd.to_datetime(data['video_date'])
data['last_updated_at'] = pd.to_datetime(data['last_updated_at'])

In [9]:
# Sort by comment date for time series analysis
data = data.sort_values('comment_date').reset_index(drop=True)

print("Date columns converted:")
print(f"  Comment date range: {data['comment_date'].min()} to {data['comment_date'].max()}")
print(f"  Video date range: {data['video_date'].min()} to {data['video_date'].max()}")
print(f"  Total time span: {(data['comment_date'].max() - data['comment_date'].min()).days} days")

Date columns converted:
  Comment date range: 2026-02-10 16:01:08 to 2026-02-26 15:16:42
  Video date range: 2026-02-10 16:00:41 to 2026-02-26 14:38:00
  Total time span: 15 days


## 2. Defining Features

### 2.1. Event Mention Detection

Based on Arc Raiders timeline and major announcements:
- **GenAI Voice Announcement**: When Embark Studios announced AI-generated voices
- **Business Model Change**: When they announced changing from free-to-play to paid
- **Game Launch**: Official release date

These dates will be used to calculate temporal features and decay functions.

In [13]:
MAJOR_EVENTS = {
    'game_announcement': pd.Timestamp('2021-12-10'),
    'genai_announcement': pd.Timestamp('2024-05-15'),
    'business_model_change': pd.Timestamp('2024-08-20'),
    'game_launch': pd.Timestamp('2025-10-30'),
    'early_access': pd.Timestamp('2025-11-15'),
}
print("Major Event Timeline:")
print("=" * 60)
for event, date in MAJOR_EVENTS.items():
    print(f"  {event:25s}: {date.strftime('%Y-%m-%d')}")
print("=" * 60)

Major Event Timeline:
  game_announcement        : 2021-12-10
  genai_announcement       : 2024-05-15
  business_model_change    : 2024-08-20
  game_launch              : 2025-10-30
  early_access             : 2025-11-15


Create binary features to detect if comments mention specific events.
This helps us identify which comments are directly responding to announcements.

In [ ]:
EVENT_KEYWORDS = {
    'game_announcement': [
        'announcement', 'announced', 'reveal', 'revealed', 'teaser', 'teased',
        'first look', 'sneak peek', 'game announcement', 'game reveal'
    ],
    'genai': [
        'ai voice', 'ai-generated', 'artificial intelligence', 'generated voice',
        'ai audio', 'synthetic voice', 'ai narration', 'ai dialogue',
        'voice ai', 'ai-voiced', 'ai acting', 'machine voice'
    ],
    'business_model': [
        'free to play', 'f2p', 'freemium', 'paid game', 'pay to play', 'p2p',
        'price', 'cost', 'buying', 'purchase', 'buy the game', 'not free',
        'charging', 'monetization', 'business model', 'payment model'
    ],
    'launch': [
        'launch', 'release', 'released', 'launching', 'came out', 'available',
        'live', 'went live', 'early access'
    ]
}

In [15]:
def detect_event_mention(text, keywords):
    """Detect if text mentions any of the keywords"""
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in keywords)

# Create event mention columns
data['mentions_genai'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['genai'])
)
data['mentions_business_model'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['business_model'])
)
data['mentions_launch'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['launch'])
)

In [16]:
# Summary of event mentions
print("Event Mention Detection Results:")
print("=" * 60)
print(f"Comments mentioning GenAI:           {data['mentions_genai'].sum():>8,} ({data['mentions_genai'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Business Model:  {data['mentions_business_model'].sum():>8,} ({data['mentions_business_model'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Launch:          {data['mentions_launch'].sum():>8,} ({data['mentions_launch'].mean()*100:>5.2f}%)")
print(f"Comments mentioning any event:       {(data['mentions_genai'] | data['mentions_business_model'] | data['mentions_launch']).sum():>8,}")
print("=" * 60)

Event Mention Detection Results:
Comments mentioning GenAI:                 16 ( 0.02%)
Comments mentioning Business Model:       506 ( 0.78%)
Comments mentioning Launch:             1,063 ( 1.64%)
Comments mentioning any event:          1,560


### 2.2. Event Proximity Windows

Create categorical features for time windows around events:
- Pre-event (1-7 days before)
- During event (day of event ± 1 day)
- Post-event windows (1-7 days, 1-4 weeks, 1-3 months after)

In [ ]:
def assign_event_window(days_from_event):
    """Assign comments to time windows relative to event"""
    if days_from_event < -7:
        return 'pre_event_far'
    elif -7 <= days_from_event < -1:
        return 'pre_event_near'
    elif -1 <= days_from_event <= 1:
        return 'during_event'
    elif 1 < days_from_event <= 7:
        return 'post_event_week1'
    elif 7 < days_from_event <= 14:
        return 'post_event_week2'
    elif 14 < days_from_event <= 30:
        return 'post_event_month1'
    elif 30 < days_from_event <= 90:
        return 'post_event_months2-3'
    else:
        return 'post_event_far'

In [ ]:
for event_name in MAJOR_EVENTS.keys():
    days_col = f'days_from_{event_name}'
    window_col = f'window_{event_name}'
    if days_col in data.columns:
        data[window_col] = data[days_col].apply(assign_event_window)

Temporal Distance Features Created:

game_announcement:
  Comments before event:        0
  Comments after event:    64,975

genai_announcement:
  Comments before event:        0
  Comments after event:    64,975

business_model_change:
  Comments before event:        0
  Comments after event:    64,975

game_launch:
  Comments before event:        0
  Comments after event:    64,975

early_access:
  Comments before event:        0
  Comments after event:    64,975
